## Training Skip-Gram Model

This notebooks is a part of [AI for Beginners Curriculum](http://aka.ms/ai-beginners)

In this example, we will look at training Skip-Gram language model to get our own Word2Vec embedding space. We will use AG News dataset as the source of text.

**Skip-Gram vs CBoW**: While CBoW predicts the center word from context words, Skip-Gram does the opposite - it predicts context words from the center word. This often leads to better embeddings for infrequent words.

In [13]:
import torch
import torchtext
import os
import collections
import builtins
import random
import numpy as np

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

First let's load our dataset and define tokenizer and vocabulary. We will set `vocab_size` to 5000 to limit computations a bit.

In [15]:
def load_dataset(ngrams = 1, min_freq = 1, vocab_size = 5000 , lines_cnt = 500):
    import csv
    tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
    print("Loading dataset...")
    
    # Read data directly from local CSV files
    data_path = '/home/ubuntu/AI-For-Beginners/lessons/5-NLP/15-LanguageModeling/lab/data/ag_news_csv'
    
    train_dataset = []
    test_dataset = []
    
    with open(f'{data_path}/train.csv', 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            # Format: class, title, description
            train_dataset.append((int(row[0]), row[1] + ' ' + row[2]))
    
    with open(f'{data_path}/test.csv', 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            test_dataset.append((int(row[0]), row[1] + ' ' + row[2]))
    
    classes = ['World', 'Sports', 'Business', 'Sci/Tech']
    print(f'Loaded {len(train_dataset)} train, {len(test_dataset)} test samples')
    
    print('Building vocab...')
    counter = collections.Counter()
    for i, (_, line) in enumerate(train_dataset):
        counter.update(torchtext.data.utils.ngrams_iterator(tokenizer(line),ngrams=ngrams))
        if i == lines_cnt:
            break
    vocab = torchtext.vocab.Vocab(collections.Counter(dict(counter.most_common(vocab_size))), min_freq=min_freq)
    print(f'Vocab size: {len(vocab)}')
    return train_dataset, test_dataset, classes, vocab, tokenizer

In [16]:
train_dataset, test_dataset, _, vocab, tokenizer = load_dataset()

Loading dataset...
Loaded 120000 train, 7600 test samples
Building vocab...
Vocab size: 5002


In [17]:
def encode(x, vocabulary, tokenizer = tokenizer):
    return [vocabulary[s] for s in tokenizer(x)]

## Skip-Gram Model

Skip-Gram learns to predict neighboring words from the center word. For example, from the sentence *I like to train networks*, with window size 2, we would get the following pairs: (I, like), (I, to), (like, I), (like, to), (like, train), (to, I), (to, like), (to, train), (to, networks), (train, like), (train, to), (train, networks), (networks, to), (networks, train). Here, the first word is the center word used as input, and the second word is a neighboring word we are predicting.

The architecture of Skip-Gram network is the following:

* Input word is passed through the embedding layer. This very embedding layer would be our Word2Vec embedding.
* Embedding vector would then be passed to a linear layer that will predict output word.

For the output, if we use `CrossEntropyLoss` as loss function, we would also have to provide just word numbers as expected results, without one-hot encoding.

In [18]:
vocab_size = len(vocab)

embedder = torch.nn.Embedding(num_embeddings = vocab_size, embedding_dim = 30)
model = torch.nn.Sequential(
    embedder,
    torch.nn.Linear(in_features = 30, out_features = vocab_size),
).to(device)

print(model)

Sequential(
  (0): Embedding(5002, 30)
  (1): Linear(in_features=30, out_features=5002, bias=True)
)


## Preparing Training Data

Now let's program the main function that will compute Skip-Gram word pairs from text. This function will allow us to specify window size, and will return a set of pairs - input (center word) and output (neighboring word).

**Key difference from CBoW**: In Skip-Gram, we predict context words from the center word, so the pairs are (center_word, context_word), which is the **reverse** of CBoW pairs.

In [19]:
def to_skipgram(sent,window_size=2):
    res = []
    for i,x in enumerate(sent):
        for j in range(max(0,i-window_size),min(i+window_size+1,len(sent))):
            if i!=j:
                # Skip-gram: input=center word, output=context word
                res.append([x, sent[j]])
    return res

print(to_skipgram(['I','like','to','train','networks']))
print(to_skipgram(encode('I like to train networks', vocab)))

[['I', 'like'], ['I', 'to'], ['like', 'I'], ['like', 'to'], ['like', 'train'], ['to', 'I'], ['to', 'like'], ['to', 'train'], ['to', 'networks'], ['train', 'like'], ['train', 'to'], ['train', 'networks'], ['networks', 'to'], ['networks', 'train']]
[[61, 92], [61, 6], [92, 61], [92, 6], [92, 0], [6, 61], [6, 92], [6, 0], [6, 2068], [0, 92], [0, 6], [0, 2068], [2068, 6], [2068, 0]]


Let's prepare the training dataset. We will go through all news, call `to_skipgram` to get the list of word pairs, and add those pairs to `X` (center words) and `Y` (context words).

In [20]:
X = []
Y = []
for i, x in zip(range(10000), train_dataset):
    for w1, w2 in to_skipgram(encode(x[1], vocab), window_size = 5):
        X.append(w1)
        Y.append(w2)

X = torch.tensor(X)
Y = torch.tensor(Y)

We will also convert that data to one dataset, and create dataloader:

In [21]:
class SimpleIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, X, Y):
        super(SimpleIterableDataset).__init__()
        self.data = []
        for i in range(len(X)):
            self.data.append( (Y[i], X[i]) )
        random.shuffle(self.data)

    def __iter__(self):
        return iter(self.data)

In [22]:
ds = SimpleIterableDataset(X, Y)
dl = torch.utils.data.DataLoader(ds, batch_size = 256)

Now let's do the actual training. We will use `SGD` optimizer with pretty high learning rate. You can also try playing around with other optimizers, such as `Adam`.

In [23]:
def train_epoch(net, dataloader, lr = 0.01, optimizer = None, loss_fn = torch.nn.CrossEntropyLoss(), epochs = None, report_freq = 1):
    optimizer = optimizer or torch.optim.Adam(net.parameters(), lr = lr)
    loss_fn = loss_fn.to(device)
    net.train()

    for i in range(epochs):
        total_loss, j = 0, 0, 
        for labels, features in dataloader:
            optimizer.zero_grad()
            features, labels = features.to(device), labels.to(device)
            out = net(features)
            loss = loss_fn(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss
            j += 1
        if i % report_freq == 0:
            print(f"Epoch: {i+1}: loss={total_loss.item()/j}")

    return total_loss.item()/j

In [24]:
train_epoch(net = model, dataloader = dl, optimizer = torch.optim.SGD(model.parameters(), lr = 0.1), loss_fn = torch.nn.CrossEntropyLoss(), epochs = 10)

Epoch: 1: loss=5.765908491157753
Epoch: 2: loss=5.442957563333741
Epoch: 3: loss=5.3801812431158975
Epoch: 4: loss=5.347694973075511
Epoch: 5: loss=5.326851441072084
Epoch: 6: loss=5.311966004008077
Epoch: 7: loss=5.30064452407906
Epoch: 8: loss=5.291609617855832
Epoch: 9: loss=5.2841637919165345
Epoch: 10: loss=5.2778877853077955


5.2778877853077955

## Trying out Word2Vec

To use Word2Vec, let's extract vectors corresponding to all words in our vocabulary:

In [29]:
vectors = torch.stack([embedder(torch.tensor(vocab[s]).to(device)) for s in vocab.itos], 0)

Let's see, for example, how the word **Paris** is encoded into a vector:

In [30]:
paris_vec = embedder(torch.tensor(vocab['paris']).to(device))
print(paris_vec)

tensor([ 0.3593,  0.7319, -0.4746,  1.5904, -1.4564, -2.2815,  0.7341,  0.6355,
        -2.7303, -0.4869,  1.7274, -0.6886, -0.1861, -0.2365, -2.1008, -0.9688,
        -0.5603, -0.8836,  1.0350,  0.4110,  1.8561, -0.1411, -0.1311,  0.6240,
         0.7631, -0.2881,  0.6211, -0.8811,  0.1660, -1.4372], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)


It is interesting to use Word2Vec to look for synonyms. The following function will return `n` closest words to a given input.

In [31]:
def close_words(x, n = 5):
  vec = embedder(torch.tensor(vocab[x]).to(device))
  top5 = np.linalg.norm(vectors.detach().cpu().numpy() - vec.detach().cpu().numpy(), axis = 1).argsort()[:n]
  return [ vocab.itos[x] for x in top5 ]

close_words('microsoft')

['microsoft', 'industry', 'businessman', 'regional', 'technology']

In [32]:
close_words('basketball')

['basketball', 'desktops', 'captured', 'creative', 'interact']

In [33]:
close_words('funds')

['funds', 'monkeys', '#38', 'enemy', 'performance']

## Takeaway

Skip-Gram is another technique for training Word2Vec models. Compared to CBoW, Skip-Gram:
- Works better for infrequent words
- Takes longer to train (predicts multiple context words per center word)
- Often produces more accurate semantic relationships

You can experiment with different window sizes, embedding dimensions, and training data to improve the quality of embeddings.